# Quickstart L1: local semantics

**L1** is the core of GT4Py V2, with *local semantics*: fields on dimensions, relations read through the neighbor tables of the mesh in scope (`with Mesh(...)`), field and scan operators, backends and compilation, and, for distributed runs, local programs on the mesh of each rank. This quickstart walks through it with one running example, a section per topic, each with a link to the in-depth notebook. The global semantics on top of L1 are in [Quickstart L2: global semantics](quickstart_l2.ipynb); all the notebooks are listed in [0. Overview](00_overview.ipynb).

The example: heat diffusion along a row of 4 cells, `c0`-`c3`, with 5 edges, `e0`-`e4`. Edge `e_i` lies between the cells `c_{i-1}` and `c_i`; `e0` and `e4` border the outside, at `t_out = 10`. Below each cell, a column of 3 levels along the vertical axis `K`, with 4 interfaces, `Staggered[K]`.

```
          e0      e1      e2      e3      e4
outside   |  c0   |  c1   |  c2   |  c3   |   outside
t_out=10  |  10   |  20   |  40   |  30   |   t_out=10

C2E: c_i -> [e_i, e_i+1]     the left and the right edge of each cell
E2C: e_i -> [c_i-1, c_i]     the left and the right cell of each edge: partial,
                             e0 has no left cell and e4 no right cell
```

One step: the flux at each edge is the temperature of its right cell minus that of its left cell (`t_out` outside the row), and each cell gets `k = 0.25` times the flux at its right edge minus the flux at its left edge. Every printed value can be derived by hand: the comments show how.

Contents: dimensions and fields · relations and meshes · field operators · scan operators · staggered axes · backends · compilation · allocation · chains and composition · local programs · cheat sheet.

The in-depth notebooks: [1. Fields and domains](01_fields_and_domains.ipynb) · [2. Connectivities](02_connectivities.ipynb) · [3. DSL definitions and operators](03_dsl_definitions_and_operators.ipynb) · [4. Scan operators](04_scan_operators.ipynb) · [5. Processors and backends](05_processors_and_backends.ipynb) · [6. Compilation](06_compilation.ipynb) · [7. Allocation](07_allocation.ipynb) · [8. Chains and partial relations](08_chains_and_partial_relations.ipynb) · [9. Footprint algebra and pairs](09_footprint_algebra_and_pairs.ipynb) · [10. Local programs](10_local_programs.ipynb) · [11. Staggered axes](11_staggered_axes.ipynb).

## Dimensions and fields

A dimension is a class of one of the kinds: `Cell` and `Edge` are `Location`s, the points of a mesh, and the vertical `K` is an `Axis`, with offsets. A field maps each point of its domain to a value: arithmetic works point by point, on the intersection of the domains, with the missing dimensions broadcast; `f[...]` restricts a field, and `tk(K + 1)` shifts it along `K`.

In depth: [1. Fields and domains](01_fields_and_domains.ipynb).

In [1]:
import numpy as np
from concepts import Axis, Location
from implementation import NumPyArrayField


class Cell(Location): ...  # the cells of the row, c0-c3


class Edge(Location): ...  # the edges, e0-e4: e_i between c_{i-1} and c_i


class K(Axis): ...  # the vertical levels, from the top down


t = NumPyArrayField({Cell: range(4)}, [10.0, 20.0, 40.0, 30.0])  # a temperature per cell
levels = NumPyArrayField({K: range(3)}, [0.0, 1.0, 2.0])  # 1 degree warmer per level down
tk = t + levels  # broadcast: the dimensions of t, then K; tk at (c, k) is t at c, plus k
print("t           =", t)
print("tk          =", tk)
print("top level   =", tk[K.at(0) : K.at(1)])  # the level K(0): t itself
print("c1 and c2   =", t[{Cell: (1, 2)}])  # an index set of cells
print("tk(K + 1)   =", tk(K + 1))  # at K(i), the value at K(i + 1): the domain shifts to K[-1:2]
print("tk(K + 1) - tk =", tk(K + 1) - tk)  # on the intersection, K[0:2]: 1 degree everywhere
try:
    Cell + 1  # a `Location` has no offsets: its neighbors come from relations
except TypeError as error:
    print("TypeError:", error)

t           = Field(Cell[0:4]: [10.0, 20.0, 40.0, 30.0])
tk          = Field(Cell[0:4], K[0:3]: [[10.0, 11.0, 12.0], [20.0, 21.0, 22.0], [40.0, 41.0, 42.0], [30.0, 31.0, 32.0]])
top level   = Field(Cell[0:4], K[0:1]: [[10.0], [20.0], [40.0], [30.0]])
c1 and c2   = Field(Cell{1, 2}: [20.0, 40.0])
tk(K + 1)   = Field(Cell[0:4], K[-1:2]: [[10.0, 11.0, 12.0], [20.0, 21.0, 22.0], [40.0, 41.0, 42.0], [30.0, 31.0, 32.0]])
tk(K + 1) - tk = Field(Cell[0:4], K[0:2]: [[1.0, 1.0], [1.0, 1.0], [1.0, 1.0], [1.0, 1.0]])
TypeError: 'Cell' has no offsets: only `Axis` dimensions have them.


## Relations and meshes

A relation, e.g. `E2C`, names a connectivity: its table, a row of neighbors for each source point, comes from the mesh of the innermost `with Mesh(...)` block, so the same code runs on any mesh. `t(E2C)` reads `t` through the table (a *gather*), adding the neighbor dimension `Neighbor[E2C]`; since `E2C` is partial, `t(E2C.boundary(t_out))` reads `t_out` for the missing cells. `sum_over`, `max_over` and `min_over` reduce over a neighbor dimension.

In depth: [2. Connectivities](02_connectivities.ipynb), [8. Chains and partial relations](08_chains_and_partial_relations.ipynb).

In [2]:
from concepts import MISSING_VALUE, Mesh, Neighbor, Relation
from implementation import max_over, neighbor_table, sum_over


class C2E(Relation[Cell, Edge], max_neighbors=2): ...  # [left edge, right edge]


class E2C(Relation[Edge, Cell], max_neighbors=2, partial=True): ...  # [left cell, right cell]


M = MISSING_VALUE  # e0 has no left cell, e4 no right cell
tables = {
    C2E: neighbor_table(C2E, [[0, 1], [1, 2], [2, 3], [3, 4]]),
    E2C: neighbor_table(E2C, [[M, 0], [0, 1], [1, 2], [2, 3], [3, M]]),
}
row = Mesh(tables, extents={Cell: 4, Edge: 5})
t_out = 10.0  # the temperature outside the row, beyond e0 and e4
with row:
    print("t(E2C)   =", t(E2C))  # only e1-e3: e0 and e4 miss a cell
    sides = t(E2C.boundary(t_out))  # t_out for the missing cells
    print("sides    =", sides)
    # e0: 10 + 10 = 20, e1: 10 + 20 = 30, e2: 20 + 40 = 60, e3: 40 + 30 = 70, e4: 30 + 10 = 40
    print("sum_over =", sum_over(sides, axis=Neighbor[E2C]))
    print("max_over =", max_over(sides, axis=Neighbor[E2C]))  # the warmer side of each edge
try:
    t(E2C)  # outside the `with` block: no tables
except LookupError as error:
    print("LookupError:", error)

t(E2C)   = Field(Edge[1:4], Neighbor[E2C][0:2]: [[10.0, 20.0], [20.0, 40.0], [40.0, 30.0]])
sides    = Field(Edge[0:5], Neighbor[E2C][0:2]: [[10.0, 10.0], [10.0, 20.0], [20.0, 40.0], [40.0, 30.0], [30.0, 10.0]])
sum_over = Field(Edge[0:5]: [20.0, 30.0, 60.0, 70.0, 40.0])
max_over = Field(Edge[0:5]: [10.0, 20.0, 40.0, 40.0, 30.0])
LookupError: No neighbor table for 'E2C': use `with Mesh(...)` to provide it.


## Field operators

`@field_operator()` checks that a function follows the rules of the DSL (no loops, for example) and turns it into a field operator; the annotations, `Field[Dims[...], float]`, give the dimensions of the arguments and of the result. `heat_step` calls `flux` and `divergence` like functions. The signs are *local fields*, with a neighbor dimension (`-1` for the left neighbor, `+1` for the right one), and `op[domain](...)` requests a part of the result. `__gt_extents__` records how the operator reads each argument: the backends propagate domains through these reads.

In depth: [3. DSL definitions and operators](03_dsl_definitions_and_operators.ipynb).

In [3]:
from concepts import Boundary, Dims, Field
from implementation import field_operator

# local fields: a sign per edge and neighbor, and per cell and neighbor, for "right - left"
edge_sign = NumPyArrayField({Edge: range(5), Neighbor[E2C]: range(2)}, [[-1.0, 1.0]] * 5)
cell_sign = NumPyArrayField({Cell: range(4), Neighbor[C2E]: range(2)}, [[-1.0, 1.0]] * 4)


@field_operator()
def flux(
    t: Field[Dims[Cell], float], t_out: float, sign: Field[Dims[Edge, Neighbor[E2C]], float]
) -> Field[Dims[Edge], float]:
    return sum_over(sign * t(E2C.boundary(t_out)), axis=Neighbor[E2C])  # right cell - left cell


@field_operator()
def divergence(
    f: Field[Dims[Edge], float], sign: Field[Dims[Cell, Neighbor[C2E]], float]
) -> Field[Dims[Cell], float]:
    return sum_over(sign * f(C2E), axis=Neighbor[C2E])  # right edge - left edge


@field_operator()
def heat_step(
    t: Field[Dims[Cell], float],
    t_out: float,
    k: float,
    edge_sign: Field[Dims[Edge, Neighbor[E2C]], float],
    cell_sign: Field[Dims[Cell, Neighbor[C2E]], float],
) -> Field[Dims[Cell], float]:
    return t + k * divergence(flux(t, t_out, edge_sign), cell_sign)


k = 0.25
with row:
    # e0: 10 - 10 = 0, e1: 20 - 10 = 10, e2: 40 - 20 = 20, e3: 30 - 40 = -10, e4: 10 - 30 = -20
    print("flux      =", flux(t, t_out, edge_sign))
    # c0: 10 + (10 - 0) / 4 = 12.5, c1: 20 + (20 - 10) / 4 = 22.5,
    # c2: 40 + (-10 - 20) / 4 = 32.5, c3: 30 + (-20 + 10) / 4 = 27.5
    print("heat_step =", heat_step(t, t_out, k, edge_sign, cell_sign))
    print("c1 and c2 =", heat_step[{Cell: (1, 2)}](t, t_out, k, edge_sign, cell_sign))


def name(step):  # a read step: a relation, `Boundary(relation)` or `Reduction(axis)`
    inner = step.relation if isinstance(step, Boundary) else getattr(step, "axis", None)
    return step.__name__ if inner is None else f"{type(step).__name__}({inner.__name__})"


# how heat_step reads t: as is, in `t + ...`, and through E2C (with a boundary value), then C2E
for read in heat_step.__gt_extents__["t"]:
    print("reads t:", ", ".join(map(name, read)) or "as is")


def cool_twice(t: Field[Dims[Cell], float]) -> Field[Dims[Cell], float]:
    # a loop: not in the DSL of field operators (programs have them: the local program below,
    # and L2's)
    for _ in range(2):
        t = 0.5 * t
    return t


try:
    field_operator()(cool_twice)
except TypeError as error:
    print("TypeError:", error)

flux      = Field(Edge[0:5]: [0.0, 10.0, 20.0, -10.0, -20.0])
heat_step = Field(Cell[0:4]: [12.5, 22.5, 32.5, 27.5])
c1 and c2 = Field(Cell{1, 2}: [22.5, 32.5])
reads t: as is
reads t: Boundary(E2C), Reduction(Neighbor[E2C]), C2E, Reduction(Neighbor[C2E])
TypeError: 'cool_twice' is not a valid field operator definition: the syntax `For` is not accepted.


## Scan operators

A scan computes the levels of a column one after the other, carrying a value from each level to the next: the definition of `@scan_operator(axis=K)` takes the carry and the values of one level, and returns the new carry, which is also the result on that level. Field operators call scan operators like any other operator.

In depth: [4. Scan operators](04_scan_operators.ipynb).

In [4]:
from implementation import scan_operator


@scan_operator(axis=K, forward=True, init=0.0)  # from the top, K(0), down
def heat_above(above: float, t: float) -> float:
    return above + t  # the sum of the levels above, this one included


@field_operator()
def column_heat(tk: Field[Dims[Cell, K], float], thickness: float) -> Field[Dims[Cell, K], float]:
    return heat_above(thickness * tk)  # a scan called from a field operator


# c0: 10, 10 + 11 = 21, 21 + 12 = 33; c1: 20, 41, 63; c2: 40, 81, 123; c3: 30, 61, 93
print("heat_above  =", heat_above(tk))
print("column_heat =", column_heat(tk, 2.0))  # twice as much: layers 2 thick
print("c2, bottom  =", heat_above[{Cell: (2,), K: range(2, 3)}](tk))  # 40 + 41 + 42 = 123

heat_above  = Field(Cell[0:4], K[0:3]: [[10.0, 21.0, 33.0], [20.0, 41.0, 63.0], [40.0, 81.0, 123.0], [30.0, 61.0, 93.0]])
column_heat = Field(Cell[0:4], K[0:3]: [[20.0, 42.0, 66.0], [40.0, 82.0, 126.0], [80.0, 162.0, 246.0], [60.0, 122.0, 186.0]])
c2, bottom  = Field(Cell{2}, K[2:3]: [[123.0]])


## Staggered axes

The interfaces between and around the levels of `K` are the staggered axis `Staggered[K]`: 4 interfaces for 3 levels, two sizes that are declared, never derived. A half offset reads across: `w(K - 0.5)` is, at each level, the value at the interface above it, and `w(K + 0.5)` the one below it.

In depth: [11. Staggered axes](11_staggered_axes.ipynb).

In [5]:
from concepts import Staggered

S = Staggered[K]  # the interfaces of the levels: Staggered[K](i) is above K(i)
w = NumPyArrayField({S: range(4)}, [0.0, 2.0, 4.0, 8.0])  # 4 interfaces for 3 levels
print("w(K - 0.5) =", w(K - 0.5))  # at K(i), the interface above it, Staggered[K](i)
print("w(K + 0.5) =", w(K + 0.5))  # at K(i), the interface below it, Staggered[K](i + 1)


@field_operator()
def to_levels(w: Field[Dims[Staggered[K]], float]) -> Field[Dims[K], float]:
    return 0.5 * (w(K - 0.5) + w(K + 0.5))  # the mean of the interfaces above and below


print("to_levels  =", to_levels(w))  # (0 + 2) / 2 = 1, (2 + 4) / 2 = 3, (4 + 8) / 2 = 6
print("tk + to_levels(w) =", tk + to_levels(w))  # c0: 10 + 1, 11 + 3, 12 + 6
try:
    NumPyArrayField({K: range(3), S: range(4)}, np.zeros((3, 4)))
except ValueError as error:
    print("ValueError:", error)

w(K - 0.5) = Field(K[0:4]: [0.0, 2.0, 4.0, 8.0])
w(K + 0.5) = Field(K[-1:3]: [0.0, 2.0, 4.0, 8.0])
to_levels  = Field(K[0:3]: [1.0, 3.0, 6.0])
tk + to_levels(w) = Field(Cell[0:4], K[0:3]: [[11.0, 14.0, 18.0], [21.0, 24.0, 28.0], [41.0, 44.0, 48.0], [31.0, 34.0, 38.0]])
ValueError: A domain cannot have both 'K' and 'Staggered[K]': they are the integer and the half-integer positions of one axis.


## Backends

A backend turns a DSL operator into its own operator, which computes the same values in another way: `eager` computes when called, `lazy` when the values are first used, `lazy_backward` too, propagating the requested domain backwards, `column_major` stores its results column-major, and `compiled` runs generated code. Below, two steps requested on `c0`: `lazy` computes all the cells of the first step, `lazy_backward` only `c0` and `c1`, the cells `c0` reads.

In depth: [5. Processors and backends](05_processors_and_backends.ipynb); when values are computed, and why: [Semantics of domains, values and fields](semantics_of_domains_values_and_fields.ipynb).

In [6]:
import contextlib
import io

from implementation import column_major, compiled, eager, lazy, lazy_backward

args = (t_out, k, edge_sign, cell_sign)  # the arguments of heat_step after t
with row:
    for backend in (eager, lazy, lazy_backward, column_major, compiled):
        with contextlib.redirect_stdout(io.StringIO()):  # silence the backends
            values = backend(heat_step)(t, *args).data.tolist()
        print(f"{backend}: {values}")
    # two steps, requested on c0 only: c0 reads e0 and e1, so the cells c0 and c1 of step 1
    for backend in (lazy, lazy_backward):
        print(f"--- {backend}")
        op = backend(heat_step)
        twice = op[{Cell: (0,)}](op(t, *args), *args)  # nothing computed yet
        print(twice)  # c0: 12.5 + ((22.5 - 12.5) - (12.5 - 10)) / 4 = 14.375

eager: [12.5, 22.5, 32.5, 27.5]
lazy: [12.5, 22.5, 32.5, 27.5]
lazy_backward: [12.5, 22.5, 32.5, 27.5]
column_major: [12.5, 22.5, 32.5, 27.5]
compiled: [12.5, 22.5, 32.5, 27.5]
--- lazy
[lazy] computed Cell[0:4], kept Cell[0:4]
[lazy] computed Cell[0:4], kept Cell{0}
Field(Cell{0}: [14.375])
--- lazy_backward
[lazy_backward] computed Cell{0}
[lazy_backward] computed Cell[0:2]
Field(Cell{0}: [14.375])


## Compilation

`compiled` is a GTIR compiler: `frontend_transform` parses the definition into GTIR, the intermediate representation, keeping the globals it uses, and `backend_transform` generates Python code from the GTIR. `compiled(op)` chains both, and compiles on the first call, or ahead of time with `.compile(...)`.

In depth: [6. Compilation](06_compilation.ipynb).

In [7]:
ir = compiled.frontend_transform(heat_step)  # the GTIR of heat_step
function = ir.fields["function"]
arguments = [a.fields["arg"] for a in function.fields["args"].fields["args"]]
print("GTIR:", ir.kind, "->", function.kind, function.fields["name"], arguments)
print("closure vars:", sorted(ir.fields["closure_vars"]))  # the globals it uses
generated = compiled.backend_transform(ir)  # Python code generated from the GTIR
print("\n".join(generated.source.splitlines()[1:]))  # its body: the `def` line is as defined
with row:
    generated(t, *args)  # compiled on the first call

GTIR: FieldOperator -> FunctionDef heat_step ['t', 't_out', 'k', 'edge_sign', 'cell_sign']
closure vars: ['C2E', 'Cell', 'Dims', 'E2C', 'Edge', 'Field', 'Neighbor', 'divergence', 'flux']
    return t + k * divergence(flux(t, t_out, edge_sign), cell_sign)
[compiled] compiling heat_step
[compiled] computed Field(Cell[0:4]: [12.5, 22.5, 32.5, 27.5])


## Allocation

Each backend provides an allocator, `backend.__gt_allocator__`, with the creation functions of the Array API standard (`zeros`, `empty_like`, ...) taking domains instead of shapes: fields in the memory layout of the backend, e.g. to write results into. `column_major` lays the values out column-major: the same values, other strides.

In depth: [7. Allocation](07_allocation.ipynb).

In [8]:
def layout(field):  # the memory order of the values of a field, and their strides in bytes
    flags = field.data.flags
    order = "C" if flags.c_contiguous else "F" if flags.f_contiguous else "strided"
    return f"{order} {field.data.strides}"


out = eager.__gt_allocator__.zeros({Cell: range(4)})  # a field to write into
with row, contextlib.redirect_stdout(io.StringIO()):
    out[{Cell: (1, 2)}] = eager(heat_step)(t, *args)  # only c1 and c2 are written
print("out =", out)

for backend in (eager, column_major):
    columns = backend.__gt_allocator__.empty_like(tk)  # tk, in the layout of the backend
    columns[...] = tk
    with contextlib.redirect_stdout(io.StringIO()):
        result = backend(column_heat)(columns, 2.0)
    print(
        f"{backend}: input {layout(columns)}, result {layout(result)}, c3 {result.data[3].tolist()}"
    )

out = Field(Cell[0:4]: [0.0, 22.5, 32.5, 0.0])
eager: input C (24, 8), result C (24, 8), c3 [60.0, 122.0, 186.0]
column_major: input F (8, 32), result F (8, 32), c3 [60.0, 122.0, 186.0]


## Chains and composition

Reads chain: `t(E2C)(C2E)` reads, for each cell, the cells of its edges, with one neighbor dimension per relation. `C2E @ E2C` composes two relations into one, with a neighbor position per path, and a relation declared with `stencil=Stencil(...)` gets its table derived by the mesh, e.g. `C2E2C`, the cells across the edges of each cell. `flatten` turns the neighbors of a relation without missing neighbors into its *pairs*, a sparse field without padding.

In depth: [8. Chains and partial relations](08_chains_and_partial_relations.ipynb), [9. Footprint algebra and pairs](09_footprint_algebra_and_pairs.ipynb).

In [9]:
from concepts import Stencil
from implementation import flatten, format_domain


class C2E2C(
    Relation[Cell, Cell],
    max_neighbors=2,
    partial=True,
    stencil=Stencil(C2E @ E2C, paths="exclude_self"),
): ...  # the cells across the edges of a cell: [left cell, right cell]


with row:
    chain = t(E2C)(C2E)  # the cells of the edges of each cell: only c1 and c2 have all of them
    print("chain:", format_domain(chain.domain))
    # its first row, c1: e1 to [c0, c1] = [10, 20], e2 to [c1, c2] = [20, 40]
    print("c1:", chain.data[0].tolist())
    # one neighbor per path: c0, through e0 to [M, c0], then through e1 to [c0, c1]
    print("C2E @ E2C =", row[C2E @ E2C].data.tolist())
    print("C2E2C     =", row[C2E2C].data.tolist())  # the paths without the cell itself
    # c0: (10 + 20) / 2 = 15, c1: (10 + 40) / 2 = 25, c2: (20 + 30) / 2 = 25, c3: (40 + 10) / 2 = 25
    print("neighbors' mean =", 0.5 * sum_over(t(C2E2C.boundary(t_out)), axis=Neighbor[C2E2C]))
    # the flux at each edge of each cell, flattened to the 8 (cell, edge) pairs of C2E
    print("pairs =", flatten(flux(t, t_out, edge_sign)(C2E), axis=Neighbor[C2E]))

chain: Cell[1:3], Neighbor[C2E][0:2], Neighbor[E2C, Neighbor[C2E]][0:2]
c1: [[10.0, 20.0], [20.0, 40.0]]
C2E @ E2C = [[-2, 0, 0, 1], [0, 1, 1, 2], [1, 2, 2, 3], [2, 3, 3, -2]]
C2E2C     = [[-2, 1], [0, 2], [1, 3], [2, -2]]
neighbors' mean = Field(Cell[0:4]: [15.0, 25.0, 25.0, 25.0])
pairs = Field(Pairs[C2E][0:8]: [0.0, 10.0, 10.0, 20.0, 20.0, -10.0, -10.0, -20.0])


## Local programs

In a distributed run, each rank sees its part of the mesh, a `LocalMesh` from `decompose`: the points it owns, then a halo, and its part of each *region*, a named set of points such as the cells of a boundary (this row has none; L2 adds two). A `@local_program` runs on every rank (here, threads) with `run_local`: `exchange` fills the halo with the values of the owners, results are requested on `Owned(Cell)`, and `global_sum` adds the owned values of all the ranks, in the order of the global indices. `gather_owned` puts the results of the ranks together: the values of the global run.

In depth: [10. Local programs](10_local_programs.ipynb).

In [10]:
from concepts import Owned
from implementation import (
    decompose,
    exchange,
    gather_owned,
    global_sum,
    local_program,
    run_local,
    scatter,
)

# 2 ranks: rank 0 owns c0, c1, e0, e1; rank 1 owns c2, c3, e2, e3, e4 (each edge goes with its
# right cell, e4 with c3); the halo: what C2E and E2C reach from them in 2 hops
owner = {Cell: [0, 0, 1, 1], Edge: [0, 0, 1, 1, 1]}
ranks = decompose(tables, {Cell: 4, Edge: 5}, {}, owner, support=(C2E, E2C), hops=2)  # no regions
for local_mesh in ranks:  # the global indices of the local points: owned first, then the halo
    print(f"rank {local_mesh.rank}:", {d.__name__: i for d, i in local_mesh.global_index.items()})


@local_program
def simulate(t, t_out, k, edge_sign, cell_sign, steps):
    edge_sign = exchange(edge_sign)  # its halo, once
    for _ in range(steps):
        t = exchange(t)  # the halo from the owners, before each step
        t = heat_step[Owned(Cell)](t, t_out, k, edge_sign, cell_sign)
    return t, global_sum(t, domain=Owned(Cell))


per_rank = [
    (scatter(t, r), t_out, k, scatter(edge_sign, r), scatter(cell_sign, r), 2) for r in ranks
]
results = run_local(simulate, ranks, per_rank)
# step 2, from 12.5, 22.5, 32.5, 27.5 (fluxes 2.5, 10, 10, -5, -17.5): c0: 12.5 + 7.5 / 4 = 14.375,
# c1: 22.5 + 0 / 4 = 22.5, c2: 32.5 - 15 / 4 = 28.75, c3: 27.5 - 12.5 / 4 = 24.375; total 90:
# each step loses k * (flux at e0 - flux at e4) = (0 + 20) / 4 = 5, then (2.5 + 17.5) / 4 = 5
gathered = gather_owned([r[0] for r in results], ranks)
print("gathered =", gathered, "| totals:", [r[1] for r in results])
with row:
    print("global   =", heat_step(heat_step(t, *args), *args))

rank 0: {'Cell': (0, 1, 2), 'Edge': (0, 1, 2)}
rank 1: {'Cell': (2, 3, 1), 'Edge': (2, 3, 4, 1)}
gathered = Field(Cell[0:4]: [14.375, 22.5, 28.75, 24.375]) | totals: [90.0, 90.0]
global   = Field(Cell[0:4]: [14.375, 22.5, 28.75, 24.375])


## Cheat sheet

| I want to... | Write... | In depth |
|---|---|---|
| declare dimensions | `class Cell(Location): ...`, `class K(Axis): ...` | [1](01_fields_and_domains.ipynb) |
| make a field | `NumPyArrayField({Cell: range(4)}, [10.0, 20.0, 40.0, 30.0])` | [1](01_fields_and_domains.ipynb) |
| restrict, shift | `t[{Cell: (1, 2)}]`, `tk[K.at(0) : K.at(1)]`, `tk(K + 1)` | [1](01_fields_and_domains.ipynb) |
| declare a relation | `class E2C(Relation[Edge, Cell], max_neighbors=2, partial=True): ...` | [2](02_connectivities.ipynb), [8](08_chains_and_partial_relations.ipynb) |
| provide the tables | `with Mesh({E2C: neighbor_table(E2C, rows), ...}, extents={...}):` | [2](02_connectivities.ipynb) |
| read the neighbors | `t(E2C)`, `t(E2C.boundary(t_out))` | [2](02_connectivities.ipynb), [8](08_chains_and_partial_relations.ipynb) |
| reduce over the neighbors | `sum_over(f, axis=Neighbor[E2C])`, `max_over(...)`, `min_over(...)` | [2](02_connectivities.ipynb) |
| write an operator | `@field_operator()` on a function annotated with `Field[Dims[Cell], float]` | [3](03_dsl_definitions_and_operators.ipynb) |
| compute a part of the result | `heat_step[{Cell: (1, 2)}](...)` | [3](03_dsl_definitions_and_operators.ipynb) |
| compute level by level | `@scan_operator(axis=K, forward=True, init=0.0)` | [4](04_scan_operators.ipynb) |
| read the interfaces | `Staggered[K]`, `w(K - 0.5)`, `w(K + 0.5)` | [11](11_staggered_axes.ipynb) |
| run on a backend | `lazy_backward(heat_step)[{Cell: (0,)}](...)` | [5](05_processors_and_backends.ipynb) |
| look at the GTIR | `compiled.frontend_transform(heat_step)` | [6](06_compilation.ipynb) |
| allocate a field | `backend.__gt_allocator__.zeros({Cell: range(4)})` | [7](07_allocation.ipynb) |
| chain and compose relations | `t(E2C)(C2E)`, `C2E @ E2C`, `stencil=Stencil(C2E @ E2C, ...)`, `flatten` | [8](08_chains_and_partial_relations.ipynb), [9](09_footprint_algebra_and_pairs.ipynb) |
| run on ranks | `decompose`, `@local_program`, `exchange`, `Owned(Cell)`, `global_sum`, `run_local`, `gather_owned` | [10](10_local_programs.ipynb) |

Next: [Quickstart L2: global semantics](quickstart_l2.ipynb), the same example on a logical mesh, with global programs.